# Interacting with the Internet in Python

In the previous lecture, we downloaded our example data file by using Jupyter Notebooks and a shell command (`curl`) like this:

In [ ]:
!curl https://data.seattle.gov/resource/v5tj-kqhc.csv -o parks.csv

This is also how we have downloaded files throughout this course, including an assignment in which we created a bash script to download the data for us.

However, Python can also make Internet requests to download or upload data. Why might we choose to use Python to download data instead of a bash script or `curl` command?

While we *could* address all of these scenarios with bash - it is a complete programming language, after all - bash is really not suited to more complex processing. Instead, it is more appropriate to use a more ergonomic programming lanuage like Python


----------------------

## Making Internet requests

Python's built in `urllib.request` module ([link to documentation](https://docs.python.org/3/library/urllib.request.html)) is the most basic tool that we can use to make requests to the Internet. Python's documentation even has [a full guide](https://docs.python.org/3/howto/urllib2.html) for working with it. We start by importing the module:

In [ ]:
import urllib.request

As always in Jupyter notebooks, we can learn more about the module with tab completion and the `?` character. Try it:

In [ ]:
urllib.request.

The most useful functions in the module will probably be:
* `urllib.request.urlretrieve`
* `urllib.request.urlopen`

#### `urllib.request` like curl

We can do the same thing we did with `curl` before and save the content to a file with `urlretrieve`:

In [ ]:
# Define a variable to store the URL we want to download
url = 'https://data.seattle.gov/resource/v5tj-kqhc.csv'

file_name_to_save = 'parks_python.csv'

# Download the content of the URL to a particular file
urllib.request.urlretrieve(url, file_name_to_save)

# Examine the file to show that it worked


#### Other features

We won't dive into all the features offered by `urllib.request`, but it does support more complex Internet request configuration, including:

* headers (both request and response)
* response codes
* request content (aka `body`) for `POST`/`PUT` requests, like form submission or file uploads
* errors
* authentication
* timeouts

HOWEVER, using these features becomes a fairly heavy-weight and manual process. If you need to do more complex Internet operations in future courses or jobs, you should prefer the `requests` library.

## The `requests` library

[The `requests` library](https://requests.readthedocs.io/en/latest/) is a third-party package that has more user-friendly configuration of advanced options (and is even recommended by the official Python documentation!). You can download and use the library like this:

In [ ]:
!conda install -y requests

In [ ]:
import requests

In [ ]:
result = requests.get(url)

result.text.split('\n')[0]

Note that this is different behavior from the `urllib.request.urlretrieve` function that we used before. Instead of saving to a file, we now have the response directly and can operate on it in Python without saving to a file.

The library also has a few useful utilities that make working with it easier, such as:

A json decoder, which translates JSON into a Python dictionary:

In [ ]:
json_url = 'https://data.seattle.gov/resource/v5tj-kqhc.json'
result = requests.get(json_url)
result.json()

Iterating through the resulting data line by line:

In [ ]:
result = requests.get(url)
for line in result.iter_lines():
    print(line)

This would probably be better written as:

In [ ]:
for line in result.iter_lines():
    print(line.decode(result.encoding))

Note that without the `decode` piece, each line is by default a "bytes" object - raw sets of 0s and 1s that we need to translate into a string with some kind of encoding.

## Exercise

We'll work again with [the Seattle pet license data from Seattle's Open Data Program](https://data.seattle.gov/City-Administration/Seattle-Pet-Licenses/jguv-t9rb/about_data). In a previous week, we saw that we could only download 1000 rows at once, and they all pertained to cats - although there's more than 38k rows in the dataset! We need to load the data in batches ("pages") so that we can put it all in a single data file.

The URL that we'll use looks like this (with our DATA 598B app token embedded in it):

```
https://data.seattle.gov/api/v3/views/jguv-t9rb/query.csv?app_token=XmnHk9K8KFspJPLY7pf6uuuAz&pageSize=10&pageNumber=1
```

That will download the first "page" of 10 results. You can move from one page to another by changing the value of the last part of the url, ie `pageNumber=2` instead of `pageNumber=1`.

Write Python code that downloads the first 10 pages of size 10 and prints them out. Note that each page will have its own header row with the column names, and you'll only want to include that once (as part of the first page).

### Exercise bonus:

Extend your solution to the exercise:
* Instead of printing, modify your code to save the data to a file called `pets.csv`.
* Instead of loading only the first 10 pages, keep making requests until you download the entire dataset. You can tell that you're done when you get no data rows from a request (ie only the header row). It can help to load pages of size 1000 instead of 10.
* Use a dictionary to set the url's query parameters (the stuff after the `?`) using the `requests` library ([here's the documentation](https://requests.readthedocs.io/en/latest/user/quickstart/#passing-parameters-in-urls)).

-----------

## Addendum: Alternative ways to use the `urllib.request` module

#### `urllib.request` like a file

The `urllib.request` module also allows us to specify a URL and then download it with `urlopen`, like this:

In [ ]:
# Open the URL just like a file
with urllib.request.urlopen(url) as response:
    
    # Fetch and decode the content
    content = response.read().decode("utf-8")
    
    # Examine the content
    lines = content.split('\n')
    print(f'First line: {lines[0]}')

Note a few things about the previous code:

* We've used the same `with ... as ...` construction as we did with files, and you can think of this in exactly the same way! We're "opening" a URL.
* Instead of a `File` object, the response is an [`HTTPResponse` object](https://docs.python.org/3/library/http.client.html#httpresponse-objects). But like a file, it has a [`read()` function](https://docs.python.org/3/library/http.client.html#http.client.HTTPResponse.read) to access the underlying data.
* One **difference** between HTTP responses and files: with files, we can specify the encoding (how we translate bytes into text) when we open the file, whereas HTTP responses are just pure bytes, and we need to decode them (translate them into a string) before we can interpret them as text.
    * What is a byte? A byte is eight bits. A bit is a single 0 or 1. So a byte would be something like `10110001`.

#### `urllib.request` with a temporary file

There's one more straightforward approach to using `urllib.request` to download content from the Internet: use a temporary file, which is cleaned up when Python exits. Then we can operate on it just like a file, but don't have to worry about deleting it.

In [ ]:
# Import two more standard library modules
import shutil
import tempfile

# Create a temporary file that will be deleted after the block
with tempfile.NamedTemporaryFile() as tmp_file:

    with urllib.request.urlopen(url) as response:
        # Copy the content into the temporary file
        shutil.copyfileobj(response, tmp_file)

    # Do something with the file
    with open(tmp_file.name) as file:
        print(f'First line: {file.readline()}')

# Now the file has been cleaned up!